# 35. SegFormer 구조 Ablation 학습

구조 원인 가설을 직접 건드리는 ablation을 실행합니다. 기본 셀은 config와 실행 계획을 만들고, 실제 재학습은 `RUN_*` 플래그를 켜면 수행됩니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 35-1. 기존 Chapter 2-2 개선 전략 요약

In [2]:
strategy_path = paths.ch2_2_runs_root / "strategy_comparison_summary.csv"
if strategy_path.exists():
    strategy = pd.read_csv(strategy_path)
    display(strategy)
else:
    print("strategy_comparison_summary.csv가 없습니다. 2-2장 27번 노트북을 먼저 실행하세요.")

,strategy,matched_mean_dice,heldout_color_dice,worst_combo_dice,stress_mean_dice,n_matched_rows,n_stress_rows
0,photometric_aug,0.584793,0.533107,0.111026,0.022616,720,480
1,group_balanced,0.549414,0.353908,0.056238,0.025052,720,480
2,baseline_no_aug,0.531328,0.318329,0.000000,0.000000,720,480


## 35-2. 구조 variant config 생성

In [3]:
base_config = paths.ch2_2_runs_root / "baseline_seed_repeats" / "seed_0" / "hf_config.json"
if not base_config.exists():
    raise FileNotFoundError("baseline seed_0 hf_config.json이 필요합니다.")

config_dir = paths.runs_root / "architecture_ablation" / "configs"
variant_configs = create_segformer_variant_configs(base_config, config_dir)
display(variant_configs)

,variant,config_path,updates
0,random_baseline,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,{}
1,lower_sr_ratio,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,"{""sr_ratios"": [4, 2, 1, 1]}"
2,stem_stride2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,"{""strides"": [2, 2, 2, 2], ""downsampling_rates""..."
3,low_sr_stem_stride2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,"{""sr_ratios"": [4, 2, 1, 1], ""strides"": [2, 2, ..."


## 35-3. optional input/structure ablation 재학습

In [4]:
samples = load_ch3_base_samples()
manifests = create_ch3_probe_manifests(samples, max_per_cell=None, seed=31)

RUN_INPUT_ABLATION = True
RUN_STRUCTURE_ABLATION = True
ABLATION_SEEDS = [0]
EPOCHS = 5
BATCH_SIZE = 8
LR = 1e-4

run_root = paths.runs_root / "architecture_ablation" / "runs"

if RUN_INPUT_ABLATION:
    for transform_mode in ["grayscale", "gray_world"]:
        for seed in ABLATION_SEEDS:
            run_dir = run_root / transform_mode / f"seed_{seed}"
            if (run_dir / "sample_metrics.csv").exists():
                print("skip existing:", run_dir)
                continue
            train_segformer_variant_experiment(
                manifests["train"],
                manifests["eval_matched"],
                run_dir,
                config_path=base_config,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                lr=LR,
                seed=seed,
                augment="photometric",
                transform_mode=transform_mode,
            )

if RUN_STRUCTURE_ABLATION:
    for row in variant_configs.itertuples(index=False):
        if row.variant == "random_baseline":
            continue
        for seed in ABLATION_SEEDS:
            run_dir = run_root / row.variant / f"seed_{seed}"
            if (run_dir / "sample_metrics.csv").exists():
                print("skip existing:", run_dir)
                continue
            train_segformer_variant_experiment(
                manifests["train"],
                manifests["eval_matched"],
                run_dir,
                config_path=row.config_path,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                lr=LR,
                seed=seed,
                augment="photometric",
            )

skip existing: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\runs\grayscale\seed_0
skip existing: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\runs\gray_world\seed_0
skip existing: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\runs\lower_sr_ratio\seed_0
skip existing: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\runs\stem_stride2\seed_0
skip existing: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\3장\runs\architecture_ablation\runs\low_sr_stem_stride2\seed_0


## 35-4. ablation 결과 수집

In [5]:
ablation_dir = paths.runs_root / "architecture_ablation"
summary = summarize_architecture_ablation(ablation_dir / "runs", ablation_dir)
if summary.empty:
    print("아직 실행된 ablation run이 없습니다. RUN_INPUT_ABLATION 또는 RUN_STRUCTURE_ABLATION을 켜고 실행하세요.")
else:
    display(summary)
    ablation_summary = pd.read_csv(ablation_dir / "architecture_ablation_summary.csv")
    display(ablation_summary)
    if ablation_summary["mean_dice_mean"].max() <= 0:
        print("판정: 현재 구조 ablation 학습은 모든 Dice가 0입니다. 원인 가설의 근거로 사용하지 말고, pretrained 초기화/더 긴 학습/class-balanced loss로 재설계해야 합니다.")

,run_dir,variant,seed,transform_mode,mean_dice,heldout_color_dice,scratch_dice,worst_combo_dice
0,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,gray_world,0,gray_world,0.0,0.0,0.0,0.0
1,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,grayscale,0,grayscale,0.0,0.0,0.0,0.0
2,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,low_sr_stem_stride2,0,None,0.0,0.0,0.0,0.0
3,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,lower_sr_ratio,0,None,0.0,0.0,0.0,0.0
4,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,stem_stride2,0,None,0.0,0.0,0.0,0.0


,variant,mean_dice_mean,mean_dice_std,mean_dice_count,heldout_color_dice_mean,heldout_color_dice_std,heldout_color_dice_count,scratch_dice_mean,scratch_dice_std,scratch_dice_count,worst_combo_dice_mean,worst_combo_dice_std,worst_combo_dice_count
0,gray_world,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1
1,grayscale,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1
2,low_sr_stem_stride2,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1
3,lower_sr_ratio,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1
4,stem_stride2,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1,0.0,NaN,1


판정: 현재 구조 ablation 학습은 모든 Dice가 0입니다. 원인 가설의 근거로 사용하지 말고, pretrained 초기화/더 긴 학습/class-balanced loss로 재설계해야 합니다.
